# NLP - Catalogo KaBuM (Memoria RAM)

Lógica: limpar o texto, remover stopwords, construir uma representacao Bag of Words e analisar as palavras mais frequentes.

In [6]:
import pandas as pd

df = pd.read_csv("data/kabum_memoria_ram_2026-04-17.csv", sep=",")
df.head()

,data_coleta,id,nome,preco_original,preco_atual,preco_pix,desconto_pct,avaliacao,num_avaliacoes,disponivel,fabricante,categoria,garantia,frete_gratis,url
0,2026-04-17,922165,"Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4,...",858.81,858.81,729.99,15,5,199,True,Husky,Hardware/Memória RAM/DDR 4/3200MHz,3 anos de garantia (3 meses de garantia legal ...,False,https://www.kabum.com.br/produto/922165/memori...
1,2026-04-17,172365,"Memória RAM Kingston Fury Beast, 8GB, 3200MHz,...",823.52,823.52,699.99,15,5,1490,True,Kingston,Hardware/Memória RAM/DDR 4,10 anos de garantia (3 meses de garantia legal...,False,https://www.kabum.com.br/produto/172365/memori...
2,2026-04-17,172366,"Memória RAM Kingston Fury Beast, 16GB, 3200MHz...",1058.81,1058.81,899.99,15,5,1046,True,Kingston,Hardware/Memória RAM/DDR 4,10 anos de garantia (3 meses de garantia legal...,False,https://www.kabum.com.br/produto/172366/memori...
3,2026-04-17,922166,"Memória RAM Husky Impulse, 16GB, 3200MHz, DDR4...",1176.46,1176.46,999.99,15,5,60,True,Husky,Hardware/Memória RAM/DDR 4/3200MHz,3 anos de garantia (3 meses de garantia legal ...,False,https://www.kabum.com.br/produto/922166/memori...
4,2026-04-17,911804,"Memória para Notebook Rise Mode Value, 8GB, 32...",0.00,388.88,349.99,10,5,9,True,Rise Mode,Hardware/Memória RAM/DDR 4/3200MHz,1 ano de garantia (3 meses de garantia legal +...,False,https://www.kabum.com.br/produto/911804/memori...


In [7]:

print(f"Antes: {len(df)} produtos")

df = df.drop_duplicates(subset="id").reset_index(drop=True)

print(f"Depois: {len(df)} produtos")

df.head()

Antes: 1145 produtos
Depois: 1123 produtos


,data_coleta,id,nome,preco_original,preco_atual,preco_pix,desconto_pct,avaliacao,num_avaliacoes,disponivel,fabricante,categoria,garantia,frete_gratis,url
0,2026-04-17,922165,"Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4,...",858.81,858.81,729.99,15,5,199,True,Husky,Hardware/Memória RAM/DDR 4/3200MHz,3 anos de garantia (3 meses de garantia legal ...,False,https://www.kabum.com.br/produto/922165/memori...
1,2026-04-17,172365,"Memória RAM Kingston Fury Beast, 8GB, 3200MHz,...",823.52,823.52,699.99,15,5,1490,True,Kingston,Hardware/Memória RAM/DDR 4,10 anos de garantia (3 meses de garantia legal...,False,https://www.kabum.com.br/produto/172365/memori...
2,2026-04-17,172366,"Memória RAM Kingston Fury Beast, 16GB, 3200MHz...",1058.81,1058.81,899.99,15,5,1046,True,Kingston,Hardware/Memória RAM/DDR 4,10 anos de garantia (3 meses de garantia legal...,False,https://www.kabum.com.br/produto/172366/memori...
3,2026-04-17,922166,"Memória RAM Husky Impulse, 16GB, 3200MHz, DDR4...",1176.46,1176.46,999.99,15,5,60,True,Husky,Hardware/Memória RAM/DDR 4/3200MHz,3 anos de garantia (3 meses de garantia legal ...,False,https://www.kabum.com.br/produto/922166/memori...
4,2026-04-17,911804,"Memória para Notebook Rise Mode Value, 8GB, 32...",0.00,388.88,349.99,10,5,9,True,Rise Mode,Hardware/Memória RAM/DDR 4/3200MHz,1 ano de garantia (3 meses de garantia legal +...,False,https://www.kabum.com.br/produto/911804/memori...


## Trabalhando apenas com o nome

Por enquanto, vamos trabalhar somente com a coluna `nome`, que concentra a descricao do produto (marca, modelo, capacidade, frequencia, tipo de memoria, etc.). As outras colunas, como `fabricante`, `categoria`, `preco_atual` e `avaliacao`, continuam no `DataFrame`, mas ficam de lado neste primeiro momento para focarmos no conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todos os produtos.

Exemplo: `"Memoria RAM Kingston Fury, 8GB, 3200MHz, DDR4"` vira `"memoria ram kingston fury 8gb 3200mhz ddr4"`.

In [10]:
# %pip install unidecode
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)

In [11]:
df["nome_limpo"] = df["nome"].apply(limpar_texto)

df[["nome", "nome_limpo"]].head()

,nome,nome_limpo
0,"Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4,...",memoria ram husky impulse 8gb 3200mhz ddr4 cl2...
1,"Memória RAM Kingston Fury Beast, 8GB, 3200MHz,...",memoria ram kingston fury beast 8gb 3200mhz dd...
2,"Memória RAM Kingston Fury Beast, 16GB, 3200MHz...",memoria ram kingston fury beast 16gb 3200mhz d...
3,"Memória RAM Husky Impulse, 16GB, 3200MHz, DDR4...",memoria ram husky impulse 16gb 3200mhz ddr4 cl...
4,"Memória para Notebook Rise Mode Value, 8GB, 32...",memoria para notebook rise mode value 8gb 3200...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todos os produtos.

In [13]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["nome_limpo"].apply(remover_stopwords)
df["nome_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["nome_limpo", "tokens_sem_stopwords", "nome_sem_stopwords"]].head()

,nome_limpo,tokens_sem_stopwords,nome_sem_stopwords
0,memoria ram husky impulse 8gb 3200mhz ddr4 cl2...,"[memoria, ram, husky, impulse, 8gb, 3200mhz, d...",memoria ram husky impulse 8gb 3200mhz ddr4 cl2...
1,memoria ram kingston fury beast 8gb 3200mhz dd...,"[memoria, ram, kingston, fury, beast, 8gb, 320...",memoria ram kingston fury beast 8gb 3200mhz dd...
2,memoria ram kingston fury beast 16gb 3200mhz d...,"[memoria, ram, kingston, fury, beast, 16gb, 32...",memoria ram kingston fury beast 16gb 3200mhz d...
3,memoria ram husky impulse 16gb 3200mhz ddr4 cl...,"[memoria, ram, husky, impulse, 16gb, 3200mhz, ...",memoria ram husky impulse 16gb 3200mhz ddr4 cl...
4,memoria para notebook rise mode value 8gb 3200...,"[memoria, notebook, rise, mode, value, 8gb, 32...",memoria notebook rise mode value 8gb 3200mhz d...


## 3. Bag of Words

Agora vamos transformar cada produto em um vetor de contagens: cada coluna vira uma palavra do vocabulario e cada celula mostra quantas vezes aquela palavra aparece no nome do produto.

Exemplo: `"kingston fury kingston"` teria `kingston = 2` e `fury = 1`.

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["nome_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,001,02,04gb,08k,10,100,104,10600,10600cl9s,10600r,...,z2,z238,z240,z4,z5,z6,z8,zbook,zeus,zmi
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
print(df_bow.columns.tolist())

['001', '02', '04gb', '08k', '10', '100', '104', '10600', '10600cl9s', '10600r', '1066mhz', '11', '110', '120gb', '1242', '128', '12800', '12800e', '12800r', '12800s', '128gb', '13', '1333', '13333mhz', '1333mhz', '14', '14iap', '14isk', '15', '15ac', '15arh', '15imh', '15isk', '16', '1600', '1600mhz', '16g', '16gb', '16gbgb', '16hd', '16hdr', '16i', '16wp', '17', '17000', '18', '1866', '1866mhz', '19', '19200', '1g6', '1gb', '1rx16', '1rx4', '1rx8', '1v', '1x', '1x16gb', '1x32gb', '1x8gb', '20', '2013', '2018', '2111', '21300', '2133', '2133mhz', '2133p', '22', '2327', '24', '240', '2400', '2400mhz', '2400t', '2400vn', '2420', '2421', '24iwl', '2520', '256', '25600', '256gb', '25v', '260', '262', '2666', '2666mhz', '288', '288pin', '2933y', '2g', '2g6d1', '2g6e1', '2gb', '2rx4', '2rx8', '2v', '2x', '2x16g', '2x16gb', '2x32gb', '2x48gb', '2x4gb', '2x64gb', '2x8gb', '30', '300', '3000', '3000mhz', '3040', '3046', '305', '3050', '3060', '3064', '3070', '3080', '3090', '310', '3135', '313

### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [27]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

1023 colunas removidas


,ac,acer,adaptador,adata,aio,aitek,al,alienware,alta,alto,...,xlinne,xmp,xn,xpg,xps,xserie,yoga,zbook,zeus,zmi
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [28]:
print(df_bow.columns.tolist())

['ac', 'acer', 'adaptador', 'adata', 'aio', 'aitek', 'al', 'alienware', 'alta', 'alto', 'amd', 'amoled', 'analyzer', 'ara', 'ares', 'argb', 'arktek', 'armor', 'aspire', 'assistente', 'asus', 'asuspro', 'asustor', 'atm', 'aura', 'axiom', 'azul', 'ba', 'baixo', 'ballistix', 'basics', 'beast', 'best', 'bestbattery', 'bgn', 'bgs', 'black', 'blade', 'bluecase', 'bolt', 'box', 'bpc', 'br', 'branca', 'branco', 'brazil', 'brazilpc', 'bt', 'cbo', 'ceamere', 'cer', 'chamadas', 'chip', 'chronus', 'cinza', 'cl', 'clarbk', 'cloudline', 'color', 'compativel', 'computador', 'computers', 'cor', 'corsair', 'cras', 'crc', 'crucial', 'ctd', 'ctech', 'cudimm', 'cwe', 'dahua', 'dclarbk', 'ddr', 'dekstop', 'dell', 'desempenho', 'desk', 'desktop', 'diamond', 'dimm', 'dissipador', 'dl', 'dominator', 'dtlabbk', 'dtlabrbk', 'dtlabrwh', 'dtlabwh', 'dual', 'duex', 'easy', 'ec', 'ecc', 'economia', 'eficiencia', 'elite', 'embalagem', 'empresarial', 'energetica', 'energia', 'enterprise', 'essentials', 'expo', 'fires

### Criando o DataFrame final

Agora juntamos os metadados relevantes do produto (`id`, `nome`, `fabricante`, `categoria`, `preco_atual`, `avaliacao`, `url`) com a matriz Bag of Words (com prefixo `bow_`).

In [29]:
metadados = df[["id", "nome", "fabricante", "categoria", "preco_atual", "avaliacao", "url"]].reset_index(drop=True)

bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,id,nome,fabricante,categoria,preco_atual,avaliacao,url,bow_ac,bow_acer,bow_adaptador,...,bow_xlinne,bow_xmp,bow_xn,bow_xpg,bow_xps,bow_xserie,bow_yoga,bow_zbook,bow_zeus,bow_zmi
0,922165,"Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4,...",Husky,Hardware/Memória RAM/DDR 4/3200MHz,858.81,5,https://www.kabum.com.br/produto/922165/memori...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,172365,"Memória RAM Kingston Fury Beast, 8GB, 3200MHz,...",Kingston,Hardware/Memória RAM/DDR 4,823.52,5,https://www.kabum.com.br/produto/172365/memori...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,172366,"Memória RAM Kingston Fury Beast, 16GB, 3200MHz...",Kingston,Hardware/Memória RAM/DDR 4,1058.81,5,https://www.kabum.com.br/produto/172366/memori...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,922166,"Memória RAM Husky Impulse, 16GB, 3200MHz, DDR4...",Husky,Hardware/Memória RAM/DDR 4/3200MHz,1176.46,5,https://www.kabum.com.br/produto/922166/memori...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,911804,"Memória para Notebook Rise Mode Value, 8GB, 32...",Rise Mode,Hardware/Memória RAM/DDR 4/3200MHz,388.88,5,https://www.kabum.com.br/produto/911804/memori...,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [30]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 355


bow_memoria     1067
bow_ram          434
bow_notebook     260
bow_kingston     209
bow_dell         170
bow_desktop      156
bow_preto        112
bow_gamer        104
bow_fury          95
bow_rgb           91
dtype: int64

In [31]:
frequencia_palavras.tail(10)

bow_up           1
bow_adaptador    1
bow_aitek        1
bow_al           1
bow_alta         1
bow_alto         1
bow_amoled       1
bow_analyzer     1
bow_ara          1
bow_ares         1
dtype: int64

In [32]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

memoria     1067
ram          432
notebook     260
kingston     209
dell         169
desktop      156
preto        112
gamer        104
fury          95
ecc           91
dtype: int64

In [34]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["nome", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(20)

,nome,palavras_unicas
650,"Memória Dale7, 2GB, 667Mhz, DDR2, Desktop, 1.8v",2
655,"Memória Dale7, 16GB, 2133Mhz, DDR4, Desktop, 1.2v",2
656,"Memória Oxy, 8GB, 1600MHz, DDR3",2
771,Memoria Ddr3 8GB 1600mhz Keepdata Kd16n11/8g,2
794,Memoria Kingston 16gb Ddr4 3200mhz Kvr32n22d8/...,2
795,Memoria 1gb Ddr2 800mhz Sodimm,2
421,"Memória Dale7, 2GB, 533Mhz, DDR2, Notebook, 1.8v",2
432,"Memória Kingston, 2GB, 667MHz, DDR2, PC5300 - ...",2
435,Memoria Kingston 8GB Ddr4 3200mhz Kvr32n22s8/8wp,2
33,"Memoria Kingston, 8GB, DDR3, 1333mhz - Kvr1333...",2
